In [1]:
print("Environment loaded")

Environment loaded


In [1]:
#Imports & Environment Setup
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
# openrouter_key = os.getenv("OPENROUTER_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add GROQ_API_KEY to .env"
    )

# Load configuration
from context_engineering.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "GROQ"
print("Environment loaded")
print(f" Provider: {provider}")
print(f"Project root: {project_root}")

Environment loaded
 Provider: GROQ
Project root: c:\Users\viraj\Zuu\Real_State_Agent


### RAG

In [2]:
#Import Chat Services
from context_engineering.application.chat_service.rag_service import RAGService;

print(" Chat services loaded from service layer")

 Chat services loaded from service layer


In [3]:
#Connect to Vector Store & Initialize LLM
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from context_engineering.infrastructure.llm_providers import (
    get_default_embeddings,
    get_chat_llm
)

# Use Groq for chat to reduce OpenRouter usage
llm = get_chat_llm(temperature=0, provider="groq")

# Embeddings require an OpenAI-compatible provider
embedding_provider = "openrouter"
embeddings = get_default_embeddings(provider=embedding_provider)

# Keep query embeddings short to reduce token usage
MAX_QUERY_CHARS = 600
def _compact_query(text: str) -> str:
    normalized = " ".join((text or "").split())
    if len(normalized) > MAX_QUERY_CHARS:
        return normalized[:MAX_QUERY_CHARS].rstrip()
    return normalized

print(f"LLM initialized: {CHAT_MODEL}")
print(f"Embeddings initialized: {EMBEDDING_MODEL} ({embedding_provider})")
print(f"Provider: {PROVIDER}")

# Connect to Qdrant local store
qdrant_path = VECTOR_DIR / "qdrant"
if not qdrant_path.exists():
    raise FileNotFoundError(" Run 02_chunking.ipynb first")

# qdrant-client 1.17+ removed search/search_points; map to query_points
if not hasattr(QdrantClient, "search") and hasattr(QdrantClient, "query_points"):
    def _search_compat(
        self,
        collection_name,
        query_vector,
        limit=10,
        offset=None,
        filter=None,
        params=None,
        with_payload=True,
        with_vectors=False,
        score_threshold=None,
        consistency=None,
        shard_key_selector=None,
        timeout=None,
        **kwargs,
    ):
        # Avoid duplicate kwargs that may already be provided by upstream callers
        kwargs.pop("query_filter", None)
        kwargs.pop("search_params", None)
        kwargs.pop("limit", None)
        kwargs.pop("offset", None)
        kwargs.pop("with_payload", None)
        kwargs.pop("with_vectors", None)
        kwargs.pop("score_threshold", None)
        result = self.query_points(
            collection_name=collection_name,
            query=query_vector,
            using=None,
            prefetch=None,
            query_filter=filter,
            search_params=params,
            limit=limit,
            offset=offset,
            with_payload=with_payload,
            with_vectors=with_vectors,
            score_threshold=score_threshold,
            lookup_from=None,
            consistency=consistency,
            shard_key_selector=shard_key_selector,
            timeout=timeout,
            **kwargs,
        )
        return result.points

    QdrantClient.search = _search_compat

qdrant_client = QdrantClient(path=str(qdrant_path))
vectorstore = Qdrant(
    client=qdrant_client,
    collection_name="sliding",
    embeddings=embeddings,
    content_payload_key="text"
 )

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
 )

print("✅ Connected to vector store")
print(f"📊 Collection size: {vectorstore.client.count(collection_name='sliding').count} vectors")

LLM initialized: llama-3.1-8b-instant
Embeddings initialized: openai/text-embedding-3-large (openrouter)
Provider: groq
✅ Connected to vector store
📊 Collection size: 1160 vectors


C:\Users\viraj\AppData\Local\Temp\ipykernel_21896\3395614351.py:82: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use `QdrantVectorStore` instead.
  vectorstore = Qdrant(


In [4]:
# Initialize RAG Service
rag_service = RAGService(
    retriever=retriever,
    llm=llm,
    k=TOP_K_RESULTS
)

print("RAGService initialized")
print(f" Retrieval: top-{TOP_K_RESULTS} documents")

RAGService initialized
 Retrieval: top-4 documents


In [20]:
query = _compact_query("Colombo")
docs = retriever.invoke(query)
len(docs), docs[0].page_content[:500], docs[0].metadata

(4,
 '://www.primelands.lk/portfolio-property/city/Colombo/en)\n\n[Weligama (1)](https://www.primelands.lk/portfolio-property/city/Weligama/en)\n\n[---\n\nShow More](#citiescollapse)\n\n[## Ella - Nuwara Eliya\n\nElla\n\n\n\n---\n\n\nExplore Land](https://www.primelands.lk/portfolio-property/ELLA-NUWARA-ELIYA/en)\n\n\n\n## Filters\n\nLooking For\n\nAll\n\nLand\n\nCoconut Land\n\nWater Front Land\n\nAgriculture Land\n\nPaddy field facing lands\n\nMountain view\n\nUp country lands\n\nDown south lands\n\nClose to sacred / ancient city lands\n\nClos',
 {'_id': 138, '_collection_name': 'fixed'})

In [21]:
# Generate Answer with RAG Service
USER_QUERY = "What is Primelands and what types of properties are listed"
QUERY = _compact_query(USER_QUERY)

print(f"Query: {QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(QUERY)

print(f"\n⏱  Generation time: {result['generation_time']:.2f}s")
print(f" Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")

Query: What is Primelands and what types of properties are listed

GENERATING ANSWER WITH RAG SERVICE...

⏱  Generation time: 2.32s
 Documents used: 4

ANSWER
Based on the provided sources, Primelands appears to be a real estate website that allows users to search for properties in Sri Lanka. 

The types of properties listed on Primelands include:

- Land [1]
- Coconut Land [1]
- Water Front Land [1]
- Agriculture Land [1]
- Paddy field facing lands [1]
- Mountain view [1]
- Up country lands [1]
- Down south lands [1]
- Close to sacred / ancient city lands [1]
- Close to highways [1]
- Close to Airport [1]
- Close to railway stations [1]
- Residential [1]
- Commercial [1]

It seems that Primelands lists a variety of properties, including both residential and commercial options, in different districts and cities across Sri Lanka.

EVIDENCE URLS


### CAG

In [5]:
#Import Chat Services
from context_engineering.application.chat_service.cag_service import CAGService;
from context_engineering.application.chat_service.cag_cache import CAGCache;

print(" CAG services loaded from service layer")

 CAG services loaded from service layer


In [23]:
# Cell 7: Initialize CAG Service with Semantic Cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Create semantic cache (embedder required for similarity matching)
cache = CAGCache(
    cache_dir=CACHE_DIR,
    embedder=embeddings,  # Uses same embeddings as vector store
    similarity_threshold=0.90,  # Catches paraphrased questions
    history_ttl_hours=24  # History expires after 24 hours
)

# Create CAG service
cag_service = CAGService(
    rag_service=rag_service,
    cache=cache
)

print("CAGService initialized (semantic matching)")
print(f" Cache directory: {CACHE_DIR}")
stats = cache.stats()
print(f"Cached responses: {stats['total_cached']}")
print(f"Similarity threshold: {stats['similarity_threshold']}")
print(f"History TTL: {stats['history_ttl_hours']}h")

CAGService initialized (semantic matching)
 Cache directory: c:\Users\viraj\Zuu\Real_State_Agent\data\cag_cache
Cached responses: 28
Similarity threshold: 0.9
History TTL: 24h


In [24]:
# FAQs are defined in config/faqs.yaml

from context_engineering.config import KNOWN_FAQS

print(f"Found {len(KNOWN_FAQS)} FAQs in config/faqs.yaml")
print("\n Sample FAQs:")
for faq in KNOWN_FAQS[:5]:
    print(f"   - {faq}")
print(f"   ... and {len(KNOWN_FAQS) - 5} more\n")

# Load FAQs into cache (this just registers them, doesn't generate responses yet)
loaded = cag_service.load_faqs(KNOWN_FAQS)
print(f"✅ Loaded {loaded} new FAQs into cache")

# To warm FAQs (generate responses), uncomment:
# print("\n🔥 Warming FAQs (this may take a few minutes)...")
# cag_service.warm_faqs()

print("\n💡 Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses")

Found 27 FAQs in config/faqs.yaml

 Sample FAQs:
   - What is Primelands and what types of properties are listed?
   - How do I search for land, apartments, houses, or portfolio properties?
   - Can I filter properties by location, budget, or property type?
   - Are the listings updated regularly?
   - Where can I find the price, size, and key features of a property?
   ... and 22 more

✅ Loaded 27 new FAQs into cache

💡 Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses


### CRAG

In [6]:
# Initialize CRAG Service
from context_engineering.config import CRAG_EXPANDED_K
from context_engineering.application.chat_service.crag_service import CRAGService

crag_service = CRAGService(
    retriever=retriever,
    llm=llm,
    initial_k=TOP_K_RESULTS,
    expanded_k=CRAG_EXPANDED_K
)

print("✅ CRAGService initialized")
print(f"🎯 Initial retrieval: top-{TOP_K_RESULTS}")
print(f"🔄 Corrective retrieval: top-{CRAG_EXPANDED_K}")


✅ CRAGService initialized
🎯 Initial retrieval: top-4
🔄 Corrective retrieval: top-8


In [7]:
#Interactive Inference - Ask Your Own Question

# INITIALIZE MISSING SERVICES (fallback if cells weren't run)
if 'rag_service' not in dir():
    rag_service = RAGService(retriever=retriever, llm=llm, k=TOP_K_RESULTS)
if 'cag_service' not in dir():
    cache = CAGCache(cache_dir=CACHE_DIR, embedder=embeddings, similarity_threshold=0.90)
    cag_service = CAGService(rag_service=rag_service, cache=cache)
if 'crag_service' not in dir():
    from context_engineering.config import CRAG_EXPANDED_K
    crag_service = CRAGService(retriever=retriever, llm=llm, initial_k=TOP_K_RESULTS, expanded_k=CRAG_EXPANDED_K)

print("=" * 80)
print("🎤 INTERACTIVE RAG INFERENCE")
print("=" * 80)
print("\n📝 Ask your question about Nawaloka Hospital...\n")

# Input your question here
YOUR_QUESTION = input("❓ Your question: ")

print(f"\n🔍 Processing query: '{YOUR_QUESTION}'\n")
print("=" * 80)

# Run through all 3 RAG systems
results = {}

# 1. Standard RAG
print("\n1️⃣  Standard RAG")
print("-" * 80)
start = time.time()
rag_result = rag_service.generate(YOUR_QUESTION)
results['RAG'] = {
    'answer': rag_result.get('answer', 'N/A'),
    'time': rag_result.get('generation_time', rag_result.get('time', 0)),
    'docs': rag_result.get('num_docs', len(rag_result.get('evidence_urls', []))),
    'urls': rag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['RAG']['time']:.2f}s")
print(f"📄 Documents retrieved: {results['RAG']['docs']}")
print(f"\n💬 Answer:")
print(results['RAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['RAG']['urls'][:3]:
    print(f"   • {url}")

# 2. Cache-Augmented Generation
print("\n" + "=" * 80)
print("\n2️⃣  Cache-Augmented Generation (CAG)")
print("-" * 80)
cag_result = cag_service.generate(YOUR_QUESTION, use_cache=True, verbose=False)
results['CAG'] = {
    'answer': cag_result.get('answer', 'N/A'),
    'time': cag_result.get('generation_time', cag_result.get('time', 0)),
    'docs': cag_result.get('num_docs', cag_result.get('docs_used', len(cag_result.get('evidence_urls', [])))),
    'cache_hit': cag_result.get('cache_hit', False),
    'urls': cag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['CAG']['time']:.2f}s")
print(f"💾 Cache hit: {results['CAG']['cache_hit']}")
print(f"📄 Documents retrieved: {results['CAG']['docs']}")
print(f"\n💬 Answer:")
print(results['CAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['CAG']['urls'][:3]:
    print(f"   • {url}")

# 3. Corrective RAG
print("\n" + "=" * 80)
print("\n3️⃣  Corrective RAG (CRAG)")
print("-" * 80)
crag_result = crag_service.generate(YOUR_QUESTION, confidence_threshold=0.6, verbose=False)
results['CRAG'] = {
    'answer': crag_result.get('answer', 'N/A'),
    'time': crag_result.get('generation_time', crag_result.get('time', 0)),
    'docs': crag_result.get('docs_used', crag_result.get('num_docs', len(crag_result.get('evidence_urls', [])))),
    'confidence': crag_result.get('confidence_final', crag_result.get('confidence', 0.0)),
    'corrected': crag_result.get('correction_applied', False),
    'urls': crag_result.get('evidence_urls', [])
}
print(f"✅ Completed in {results['CRAG']['time']:.2f}s")
print(f"📊 Confidence: {results['CRAG']['confidence']:.2f}")
print(f"🔧 Correction applied: {results['CRAG']['corrected']}")
print(f"📄 Documents used: {results['CRAG']['docs']}")
print(f"\n💬 Answer:")
print(results['CRAG']['answer'])
print(f"\n📎 Evidence URLs:")
for url in results['CRAG']['urls'][:3]:
    print(f"   • {url}")

print("\n" + "=" * 80)
print("📊 PERFORMANCE COMPARISON")
print("=" * 80)
print(f"\n{'System':<15} {'Time (s)':<12} {'Docs':<8} {'Special Feature':<30}")
print("-" * 80)
print(f"{'Standard RAG':<15} {results['RAG']['time']:<12.2f} {results['RAG']['docs']:<8} {'Baseline':<30}")

# CAG feature
cag_feature = 'Cache: ' + ('HIT ⚡' if results['CAG']['cache_hit'] else 'MISS')
print(f"{'CAG':<15} {results['CAG']['time']:<12.2f} {results['CAG']['docs']:<8} {cag_feature:<30}")

# CRAG feature
crag_conf = results['CRAG']['confidence']
crag_emoji = ' ✅' if crag_conf > 0.7 else ' ⚠️'
crag_feature = f"Confidence: {crag_conf:.2f}{crag_emoji}"
print(f"{'CRAG':<15} {results['CRAG']['time']:<12.2f} {results['CRAG']['docs']:<8} {crag_feature:<30}")

# Summary recommendation
print("=" * 80)
print("💡 RECOMMENDATION")
print("=" * 80)

fastest = min(results.items(), key=lambda x: x[1]['time'])
print(f"⚡ Fastest: {fastest[0]} ({fastest[1]['time']:.2f}s)")

if results['CAG']['cache_hit']:
    print("🎯 Best Choice: CAG (cache hit = instant response)")
elif results['CRAG']['confidence'] > 0.7:
    print("🎯 Best Choice: CRAG (high confidence + corrective capability)")
else:
    print("🎯 Best Choice: Standard RAG (reliable baseline)")

print("\n✅ Inference complete!")
print("=" * 80)


🎤 INTERACTIVE RAG INFERENCE

📝 Ask your question about Nawaloka Hospital...


🔍 Processing query: 'list down apartments in colombo'


1️⃣  Standard RAG
--------------------------------------------------------------------------------
✅ Completed in 3.20s
📄 Documents retrieved: 4

💬 Answer:
Based on the provided sources, here are the apartments listed in Colombo:

1. The Colombo Border - Peliyagoda [1] [2] [3] [4]
2. Gampaha - The Palace [1] [2] [3] [4]
3. Skye Blossom - Kottawa [1] [2] [3] [4]
4. Yolo - Kiribathgoda [1] [2] [3] [4]

Note that the prices for these apartments are also mentioned in the sources:

- The Colombo Border - Peliyagoda: 35,000,000 LKR Per Unit Upwards [1] [2] [3] [4]
- Gampaha - The Palace: 22,000,000 LKR Per Unit Upwards [1] [2] [3] [4]
- Skye Blossom - Kottawa: 50,000,000 LKR Per Unit Upwards [1] [2] [3] [4]
- Yolo - Kiribathgoda: 35,000,000 LKR Per Unit Upwards [1] [2] [3] [4]

Please note that there might be other apartments in Colombo not listed in these sou